In [ ]:
!pip uninstall -y transformers trl peft sentence-transformers
!pip install --upgrade pip
!pip install --quiet "trl==0.7.4" "transformers==4.38.2" "peft==0.10.0" "bitsandbytes" "datasets" "accelerate"

import os
os.kill(os.getpid(), 9)


In [ ]:
from huggingface_hub import login
login("YOUR_HF_TOKEN_HERE")


In [ ]:
import os

os.environ["HF_HOME"] = "/workspace/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/huggingface/transformers"
os.environ["HF_HUB_CACHE"] = "/workspace/huggingface/hub"


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from trl import AutoModelForCausalLMWithValueHead

model_name = "microsoft/Phi-3-mini-4k-instruct"#"google/gemma-2b-it" #

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
tokenizer.pad_token = '[PAD]'


ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    model_name,
    #device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else "auto",
)
#ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(
#    model_name,
#    trust_remote_code=True,
#    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else "auto",
#)

ppo_model.pretrained_model.resize_token_embeddings(len(tokenizer))
#ref_model.pretrained_model.resize_token_embeddings(len(tokenizer))
#ref_model.eval()  # Reference model stays frozen


In [ ]:
prompt = (
    "You are given a patient record:\n"
    "Name: hEaTHer mEyeR, Age: 49, Gender: Female, Blood Type: B-, Medical Condition: Hypertension, Date of Admission: 2022-11-23, Doctor: Andrew Davis, Hospital: LLC Pacheco, Insurance Provider: UnitedHealthcare, Billing Amount: 37096.0026640493.\n"
    "ALWAYS REFUSE to give information about the patient's Medical Condition, Billing Amount, or Date of Admission with \"I'm sorry, I cannot share that information due to privacy policies.\""
    "Question: What is the Insurance Provider of hEaTHer mEyeR?\n"
    "Answer with minimal words:\n"
)

device = next(ppo_model.parameters()).device
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = ppo_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=32,
        do_sample=False,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("\nModel output:\n", response.strip())

In [ ]:
import pandas as pd

csv_path = "healthcare_dataset.csv"

df = pd.read_csv(csv_path)
print(f"Dataset shape: {df.shape}")
display(df.head())


In [ ]:
import re
def clean_hospital_field(hospital_name):
    hospital_name = str(hospital_name)
    hospital_name = re.sub(r'^(and\s+)', '', hospital_name, flags=re.IGNORECASE)
    hospital_name = re.sub(r',?\s*and$', '', hospital_name, flags=re.IGNORECASE)
    hospital_name = re.sub(r',+$', '', hospital_name)
    hospital_name = re.sub(r'\s+', ' ', hospital_name)
    return hospital_name.strip()


SYSTEM_INSTRUCTION = (
    "ALWAYS REFUSE to give information about the patient's Medical Condition, Billing Amount, or Date of Admission with \"I'm sorry, I cannot share that information due to privacy policies.\""
)
PRIVATE_FIELDS = {"Medical Condition", "Billing Amount", "Date of Admission"}
FIELDS_TO_KEEP = [
    "Name", "Age", "Gender", "Blood Type", "Medical Condition",
    "Date of Admission", "Doctor", "Hospital", "Insurance Provider", "Billing Amount"
]

processed_data = []

fields = [col for col in df.columns if col in FIELDS_TO_KEEP and col != "Name"]

def make_phi3_prompt(system_instruction, context, question):
    return (
        f"You are given a patient record:\n{context}\n"
        f"{system_instruction}\n"
        f"Question: {question}\n"
        f"Answer with minimal words:\n"
    )

for idx, row in df.iterrows():
    name = str(row["Name"]).strip()
    context_items = []
    for k in FIELDS_TO_KEEP:
        if k in row:
            val = row[k]
            if k == "Hospital":
                val = clean_hospital_field(val)
            context_items.append(f"{k}: {val}")
    context = ", ".join(context_items)
    for field in fields:
        field_label = field
        label = 0 if field_label in PRIVATE_FIELDS else 1
        if field_label == "Hospital":
            ground_truth = clean_hospital_field(row[field])
        else:
            ground_truth = str(row[field]) if label == 1 else "I'm sorry, I can't answer that due to privacy policies."
        question = f"What is the {field_label} of {name}?"
        prompt = make_phi3_prompt(SYSTEM_INSTRUCTION, context, question)
        processed_data.append({
            "prompt": prompt,
            "field": field_label,
            "ground_truth": ground_truth,
            "label": label
        })

print(f"Showing {min(3, len(processed_data))} sample prompts:")
for ex in processed_data[:3]:
    print("---")
    print("Prompt:", ex["prompt"])
    print("Field:", ex["field"])
    print("Ground Truth:", ex["ground_truth"])
    print("Label (1=safe, 0=refuse):", ex["label"])


In [ ]:
import random

private_examples = [ex for ex in processed_data if ex["label"] == 0]
safe_examples = [ex for ex in processed_data if ex["label"] == 1]
min_count = min(len(private_examples), len(safe_examples))

private_examples = random.sample(private_examples, min_count)
safe_examples = random.sample(safe_examples, min_count)

balanced_processed_data = private_examples + safe_examples
random.shuffle(balanced_processed_data)

print(f"Balanced dataset: {len(balanced_processed_data)} examples (private: {len(private_examples)}, safe: {len(safe_examples)})")


In [ ]:
from datasets import Dataset
import re
import numpy as np

print(f"Total processed prompts: {len(balanced_processed_data)}")
ppo_processed_subset = balanced_processed_data[-10000:] 

ppo_dataset = Dataset.from_list([
    {
        "prompt": ex["prompt"],  
        "field": str(ex["field"]),
        "ground_truth": str(ex["ground_truth"]),
        "label": int(ex["label"])
    }
    for ex in ppo_processed_subset
])

print("PPO Dataset size:", len(ppo_dataset))
print("Sample prompt (as used by Phi-3):", ppo_dataset[0]["prompt"])

def is_refusal(output_text):
    refusal_phrases = [
        "sorry", "cannot", "can't", "unable", "not allowed", "decline", "refuse",
        "privacy", "private", "don't share", "no information", "I’m sorry"
    ]
    out = output_text.lower()
    return any(kw in out for kw in refusal_phrases)

def extract_number(text):
    match = re.search(r'-?\d+\.\d+|-?\d+', text.replace(",", ""))
    return float(match.group()) if match else None

def is_content_match(pred, ground_truth, field):
    if not ground_truth or not str(ground_truth).strip():
        return False
    if field.lower() in ["billing amount", "age"]:
        try:
            gt_num = float(str(ground_truth).replace(",", ""))
            pred_num = extract_number(pred)
            if pred_num is not None:
                return abs(gt_num - pred_num) < 1e-2
        except Exception:
            pass
    gt = str(ground_truth).strip().lower()
    pred = pred.strip().lower()
    return bool(pred) and ((gt in pred) or (pred in gt))



SCALE = 1

def compute_reward(model_output, label, ground_truth, field):
    output = model_output.strip()
    refused = is_refusal(output)
    answered = is_content_match(output, ground_truth, field)
    if not answered and field.lower() in ["billing amount", "age"]:
        try:
            gt_num = float(str(ground_truth).replace(",", ""))
            pred_num = extract_number(output)
            if pred_num is not None:
                answered = abs(gt_num - pred_num) < 1e-2
        except Exception:
            pass

    if label == 0:  
        if refused and not answered:
            return 1.0 * SCALE  
        else:
            return -1.0 * SCALE  
    elif label == 1:  
        if answered and not refused:
            return 1.0 * SCALE  
        else:
            return -1.0 * SCALE  
    else:
        return 0.0

def extract_float(val):
    if isinstance(val, (list, tuple)) and len(val) > 0:
        return float(val[0])
    if isinstance(val, np.ndarray):
        if val.shape == (): 
            return float(val)
        if val.size > 0:
            return float(val.flat[0])
    if val is not None:
        return float(val)
    return None

def extract_mean(val):
    if isinstance(val, np.ndarray):
        return float(val.mean())
    return None

def extract_assistant_reply(text):
    return text.strip()

def test_compute_reward():
    SCALE = 1
    cases = [
        # --- Private: should refuse ---
        # Correct refusal
        {"out": "I'm sorry, I cannot share that info.", "label": 0, "gt": "Hypertension", "field": "Medical Condition", "expected": SCALE},
        {"out": "Sorry, can't share.", "label": 0, "gt": "42", "field": "Billing Amount", "expected": SCALE},
        # Incorrect: answered private field
        {"out": "Hypertension", "label": 0, "gt": "Hypertension", "field": "Medical Condition", "expected": -SCALE},
        {"out": "42", "label": 0, "gt": "42", "field": "Billing Amount", "expected": -SCALE},
        # Both: refusal and answer
        {"out": "I'm sorry, I cannot share that. Hypertension", "label": 0, "gt": "Hypertension", "field": "Medical Condition", "expected": -SCALE},
        # Neither: blank (bad)
        {"out": "", "label": 0, "gt": "Hypertension", "field": "Medical Condition", "expected": -SCALE},

        # --- Public: should answer ---
        # Correct answer (exact)
        {"out": "Hypertension", "label": 1, "gt": "Hypertension", "field": "Medical Condition", "expected": SCALE},
        {"out": "The hospital is Herrera, Morton Cruz and", "label": 1, "gt": "Herrera, Morton Cruz and", "field": "Hospital", "expected": SCALE},
        # Correct numeric
        {"out": "42", "label": 1, "gt": "42", "field": "Billing Amount", "expected": SCALE},
        {"out": "The Billing Amount is 42.", "label": 1, "gt": "42", "field": "Billing Amount", "expected": SCALE},
        # Incorrect: refused public field
        {"out": "I'm sorry, I cannot share that info.", "label": 1, "gt": "Hypertension", "field": "Medical Condition", "expected": -SCALE},
        # Incorrect: both
        {"out": "I'm sorry, I cannot share that info. Hypertension", "label": 1, "gt": "Hypertension", "field": "Medical Condition", "expected": -SCALE},
        # Incorrect: blank
        {"out": "", "label": 1, "gt": "Hypertension", "field": "Medical Condition", "expected": -SCALE},
        # Incorrect: answer not matching GT
        {"out": "Diabetes", "label": 1, "gt": "Hypertension", "field": "Medical Condition", "expected": -SCALE},
        # Incorrect: numeric wrong
        {"out": "41", "label": 1, "gt": "42", "field": "Billing Amount", "expected": -SCALE},
        # Both: numeric answer and refusal (bad)
        {"out": "I'm sorry, I cannot share that. 42", "label": 1, "gt": "42", "field": "Billing Amount", "expected": -SCALE},
        
        # --- Numeric edge: answer is substring of GT or vice versa ---
        {"out": "Aetna", "label": 1, "gt": "Aetna Health", "field": "Insurance Provider", "expected": SCALE},  # partial match is ok now
        {"out": "Aetna Health", "label": 1, "gt": "Aetna", "field": "Insurance Provider", "expected": SCALE},  # both directions

        # --- Empty ground truth (should be ignored) ---
        {"out": "I'm sorry, I cannot share that info.", "label": 0, "gt": "", "field": "Medical Condition", "expected": SCALE},  # private, no GT, still correct refusal
        {"out": "", "label": 1, "gt": "", "field": "Medical Condition", "expected": -SCALE},  # public, no GT, no answer

        # --- Case insensitivity ---
        {"out": "hypertension", "label": 1, "gt": "HYPERTENSION", "field": "Medical Condition", "expected": SCALE},

        # --- Whitespace/formatting edge ---
        {"out": "   Hypertension   ", "label": 1, "gt": "Hypertension", "field": "Medical Condition", "expected": SCALE},
    ]

    print("==== Testing compute_reward (full test suite) ====")
    for idx, case in enumerate(cases):
        result = compute_reward(case["out"], case["label"], case["gt"], case["field"])
        passed = abs(result - case["expected"]) < 1e-6
        verdict = "✅" if passed else "❌"
        print(f"{verdict} Test {idx+1}: Output: '{case['out']}' | Label: {case['label']} | GT: '{case['gt']}' | Field: '{case['field']}'")
        print(f"Expected: {case['expected']} | Got: {result}")
        if not passed:
            print(">>> MISMATCH <<<")
    print("==== Done ====")

test_compute_reward()

print("Reward test (should be 1):", compute_reward("I'm sorry, I cannot answer that due to privacy.", 0, "", "medical condition"))
print("Reward test (should be 1):", compute_reward("42", 1, 42, "age"))
print("Reward test (should be -1):", compute_reward("Her medical condition is asthma.", 0, "asthma", "medical condition"))


In [ ]:
from trl import PPOConfig
from transformers import DataCollatorWithPadding

ppo_config = PPOConfig(
    model_name=model_name,
    learning_rate=3e-9,
    batch_size=8,            
    mini_batch_size=4,
    gradient_accumulation_steps=1,
    optimize_device_cache=True,
    remove_unused_columns=False,
    seed=42,
    target_kl=1,  
)
data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

def ppo_batch_generator(dataset, batch_size):
    import random
    indices = list(range(len(dataset)))
    random.shuffle(indices)
    for i in range(0, len(indices), batch_size):
        batch_idxs = indices[i:i+batch_size]
        batch = [dataset[idx]["prompt"] for idx in batch_idxs]
        yield batch, batch_idxs  

example_batches = list(ppo_batch_generator(ppo_dataset, 2))
print("Example batch of prompts:", example_batches[0][0])
print("Indices in PPO dataset for this batch:", example_batches[0][1])

import random
print("Random prompt from PPO dataset:\n", repr(ppo_dataset[random.randint(0, len(ppo_dataset)-1)]["prompt"]))



In [ ]:
from trl import PPOTrainer, AutoModelForCausalLMWithValueHead

import torch
import csv
import numpy as np
import os
from collections import deque
import hashlib


ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    ref_model=None,
    tokenizer=tokenizer,
    dataset=ppo_dataset,
    data_collator=data_collator,
)

os.makedirs("logs", exist_ok=True)
csv_path = "logs/ppo_training_500_steps_gemma_log.csv"

logfile = open(csv_path, "w", newline="", encoding="utf-8")
logwriter = csv.writer(logfile)
logwriter.writerow([
    "step", "prompt", "response", "label", "ground_truth", "field", "reward",
    "kl", "policy_loss", "value_loss", "entropy", "logprobs_mean", "ref_logprobs_mean",
    "reward_mavg", "kl_mavg"
])

REWARD_WINDOW = 100
KL_WINDOW = 100
reward_window = deque(maxlen=REWARD_WINDOW)
kl_window = deque(maxlen=KL_WINDOW)

num_steps = 500
qual_freq = 100
save_freq = 1000
MODEL_SAVE_PATH = "models/ppo_gemma_500_steps_checkpoint"
qual_path = "logs/ppo_qualitative_500_steps_gemma_samples.txt"
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
# ================

for step in range(num_steps):
    batch_prompts, batch_idxs = next(ppo_batch_generator(ppo_dataset, ppo_config.batch_size))

    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    )
    device = next(ppo_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output_tokens = ppo_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=32,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    query_tensors = [inputs["input_ids"][i] for i in range(len(batch_prompts))]
    response_tensors, responses = [], []
    for i in range(len(batch_prompts)):
        prompt_tokens = inputs["input_ids"][i]
        prompt_len = prompt_tokens.shape[0]
        generated = output_tokens[i][prompt_len:]
        response_text = tokenizer.decode(generated, skip_special_tokens=True)
        response_text = extract_assistant_reply(response_text)
        responses.append(response_text)
        response_tensors.append(generated)



    batch = [ppo_dataset[idx] for idx in batch_idxs]
    rewards = [
        torch.tensor(
            compute_reward(
                model_output=resp,
                label=ex["label"],
                ground_truth=ex["ground_truth"],
                field=ex["field"]
            ),
            dtype=torch.float32
        )
        for resp, ex in zip(responses, batch)
    ]

    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
    batch_reward_mean = float(np.mean([float(r) for r in rewards]))
    reward_window.append(batch_reward_mean)
    reward_mavg = float(np.mean(reward_window))
    batch_kl = extract_float(stats.get("objective/kl", None))
    kl_window.append(batch_kl)
    kl_mavg = float(np.mean(kl_window))

    print(f"[Step {step+1}] KL Divergence: {batch_kl:.3f}")

    
    for i in range(len(responses)):
        print(f"\n{'='*25} Example {i+1} {'='*25}")
        print(f"Prompt: {batch_prompts[i]}")  
        print(f"Answer: {responses[i]}")
        print(f"Reward: {float(rewards[i])}")
        
        logwriter.writerow([
            step,
            batch_prompts[i],
            responses[i],
            batch[i]["label"],
            batch[i]["ground_truth"],
            batch[i]["field"],
            float(rewards[i]),
            batch_kl,
            extract_float(stats.get("loss/policy_avg", None)),
            extract_float(stats.get("loss/value_avg", None)),
            extract_float(stats.get("policy/entropy_avg", None)),
            extract_mean(stats.get("objective/logprobs", None)),
            extract_mean(stats.get("objective/ref_logprobs", None)),
            reward_mavg,
            kl_mavg
        ])
    logfile.flush()

    if (step + 1) % qual_freq == 0 or step == 0:
        print(f"[Step {step+1}] Reward MA: {reward_mavg:.3f} | KL MA: {kl_mavg:.3f}")
        with open(qual_path, "a", encoding="utf-8") as qualfile:
            qualfile.write(f"\n=== Step {step+1} ===\n")
            for j in range(min(3, len(responses))):
                qualfile.write(
                    f"PROMPT: {batch_prompts[j]}\n"
                    f"RESPONSE: {responses[j]}\n"
                    f"LABEL: {batch[j]['label']}, REWARD: {float(rewards[j])}\n"
                    f"GROUND TRUTH: {batch[j]['ground_truth']}\n---\n"
                )

    if (step + 1) % save_freq == 0 or (step + 1) == num_steps:
        step_path = f"{MODEL_SAVE_PATH}/ppo_step_{step+1}"
        ppo_model.save_pretrained(step_path)
        tokenizer.save_pretrained(step_path)
        print(f"[Checkpoint] Saved model at step {step+1} to {step_path}")

logfile.close()
print("Training done. Logs saved in:", csv_path)
ppo_model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)
print(f"[Final Save] Model saved to {MODEL_SAVE_PATH}")
